## 03 - Probability calibration (CalibratedClassifierCV)

Load the CatBoost model from 02; 

Use **half of the OOT test set** for calibration and the other half for evaluation to avoid leakage. 

Use **Platt scaling (sigmoid)** instead of Isotonic so ranking is preserved and ROC-AUC/PR-AUC are not distorted under severe imbalance; 

Also we saved the calibrated model.

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ROOT = Path.cwd()
MODEL_DIR = ROOT / "fraud_detection" / "models"
FEATURE_PATH = ROOT / "fraud_detection" / "data" / "features_processed.csv"

with open(MODEL_DIR / "feature_config.json") as f:
    config = json.load(f)
feature_cols = config["feature_cols"]
cat_features = config["cat_features"]

train_df = pd.read_csv(MODEL_DIR / "oot_train_index.csv")
test_df = pd.read_csv(MODEL_DIR / "oot_test_index.csv")
X_train = train_df[feature_cols]
y_train = train_df["is_fraud"]
X_test = test_df[feature_cols]
y_test = test_df["is_fraud"]
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (444575, 22) Test: (111144, 22)


## Use half of OOT test for calibration, half for evaluation (no leakage)

CatBoost in 02 was trained only on train_df and **never saw** test_df. Split test in half: first 50% to fit the calibrator, last 50% to report metrics, so both calibration and evaluation are leakage-free.

In [2]:
# Split test in half: first 50% for calibration, last 50% for evaluation (base never saw test in 02)
cal_ratio = 0.5
n_test = len(X_test)
n_cal = int(n_test * cal_ratio)
X_cal = X_test.iloc[:n_cal]
y_cal = y_test.iloc[:n_cal]
X_eval = X_test.iloc[n_cal:]
y_eval = y_test.iloc[n_cal:]
print(f"Cal size: {len(X_cal)}, Eval size: {len(X_eval)} (no leakage from train)")

Cal size: 55572, Eval size: 55572 (no leakage from train)


## Load CatBoost and wrap as CalibratedClassifierCV (sigmoid / Platt scaling)

With very few positive samples, Isotonic can flatten probabilities into steps and break ranking; **sigmoid** fits a single smooth S-curve, preserves relative ranks, and keeps ROC-AUC/PR-AUC effectively unchanged.

In [3]:
base = CatBoostClassifier()
base.load_model(str(MODEL_DIR / "catboost_fraud.cbm"))

# Calibrate on half of OOT test; method=sigmoid preserves ranking and AUC
calibrated = CalibratedClassifierCV(base, method="sigmoid", cv="prefit")
calibrated.fit(X_cal, y_cal)
print("Calibration fitted (Platt scaling).")

/Users/zhumiban/anaconda3/envs/agent_bank/lib/python3.10/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


Calibration fitted (Platt scaling).


## Compare before/after calibration on the **eval set** (last 50% of test): ROC-AUC / PR-AUC / Brier

In [4]:
# Compute metrics on eval set (last 50% of test), independent of calibration set
prob_raw = base.predict_proba(X_eval)[:, 1]
prob_cal = calibrated.predict_proba(X_eval)[:, 1]

print("Eval set (held-out half of OOT test):")
print("  ROC-AUC (raw):   ", roc_auc_score(y_eval, prob_raw))
print("  ROC-AUC (cal):   ", roc_auc_score(y_eval, prob_cal))
print("  PR-AUC (raw):    ", average_precision_score(y_eval, prob_raw))
print("  PR-AUC (cal):    ", average_precision_score(y_eval, prob_cal))
print("  Brier (raw, lower better): ", brier_score_loss(y_eval, prob_raw))
print("  Brier (cal, lower better): ", brier_score_loss(y_eval, prob_cal))

import joblib
joblib.dump(calibrated, MODEL_DIR / "catboost_calibrated.pkl")
print(f"Saved calibrated model to {MODEL_DIR / 'catboost_calibrated.pkl'}")

Eval set (held-out half of OOT test):
  ROC-AUC (raw):    0.998665319127597
  ROC-AUC (cal):    0.998665319127597
  PR-AUC (raw):     0.7253365716629149
  PR-AUC (cal):     0.7253365716629149
  Brier (raw, lower better):  0.002261070201202974
  Brier (cal, lower better):  0.0004460350196465133
Saved calibrated model to /Users/zhumiban/Desktop/agent_bank/fraud_detection/models/catboost_calibrated.pkl
